
**Sign Language Recognition - Deep Learning**


Libraries:

In [10]:
# pip install opencv-python
import cv2

Implementation:

In [11]:
def segmentation(frame, threshold=25):

    global background

    diff = cv2.absdiff(background.astype("uint8"), frame)

    _, processed_frame = cv2.threshold(
        diff,
        threshold,
        255,
        cv2.THRESH_BINARY
    )
    
    contours, _ = cv2.findContours(processed_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return None
    else :
        contours = max(contours, key=cv2.contourArea)

    return (processed_frame, contours)

In [12]:
# =========================
# Camera Initialization
# =========================
cam = cv2.VideoCapture(0, cv2.CAP_V4L2)

# =========================
# ROI (Region of Interest)
# =========================
TOP = 50
BOTTOM = 300
RIGHT = 50
LEFT = 250

# =========================
# Global Variables
# =========================
background = None
count = 0



# =========================
# Main Loop
# =========================
while True:

    value, frame = cam.read()

    # Copy frame and flip (mirror effect)
    frame_copy = frame.copy()
    frame_copy = cv2.flip(frame_copy, 1)

    # ROI extraction
    roi = frame_copy[TOP:BOTTOM, RIGHT:LEFT]

    # Preprocessing
    roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    roi_gray = cv2.GaussianBlur(roi_gray, (9, 9), 0)

    # Initialize background
    if background is None:
        background = roi_gray.copy().astype("float")

    # Draw ROI rectangle
    cv2.rectangle(frame_copy, (RIGHT, TOP), (LEFT, BOTTOM), (0, 255, 0), 2)

    # Show camera
    cv2.imshow("Camera", frame_copy)

    # Background learning (first 30 frames)
    if count < 30:
        cv2.accumulateWeighted(roi_gray, background, 0.5)

    # Hand segmentation
    collection = segmentation(roi_gray)

    if collection is not None:
        roi_processed, contours = collection    
        cv2.imshow("Processed", roi_processed)

    count += 1

    # Exit with ESC
    if cv2.waitKey(1) == 27:
        break


# =========================
# Release Resources
# =========================
cam.release()
cv2.destroyAllWindows()